# Monte Carlo Portfolio Simulation — 1-Year Forecast

> **⚠️ PAPER-ONLY, NON-ADVISORY** — This notebook is a **simulation**, not investment advice, not an order.
> No API keys, no live feeds, no order execution. Compliant with Iron Law #8 (`docs/protocols/bot-trading-iron-law.md`).

**Skill reference**: `skills/awiki/monte-carlo-quant-analysis/SKILL.md`  
**Data**: synthetic canned CSV (seeded `np.random.default_rng(42)`) — 100% reproducible, no external download.  
**Prerequisites**: `numpy`, `matplotlib` (standard scientific Python stack).

This notebook walks through the 6 `process_steps` of the skill end-to-end.

In [ ]:
# Cell 2 — Imports + canned synthetic CSV (seeded, reproducible, no download)
import numpy as np
import matplotlib.pyplot as plt
from io import StringIO

# Generate a CANNED historical-ish daily-returns CSV for 3 synthetic assets.
# Real tickers are NOT used — these are AAPL-like / MSFT-like / SPY-like in
# statistical shape only (annualized mu/sigma drawn from typical equity ranges).
# Seed = 42 → identical every run. Replace with real CannedMarketDataFeed
# output (see bot-trading-iron-law.md §Amendment 2026-06-12) when wired.
rng = np.random.default_rng(seed=42)
TRADING_DAYS = 252
ASSETS = ["ASSET_A", "ASSET_B", "ASSET_C"]  # synthetic names (not real tickers)
# annualized (mu, sigma) per asset — illustrative only, NOT fitted from real data
PARAMS = [
    (0.10, 0.18),  # ASSET_A: moderate return, moderate vol
    (0.15, 0.25),  # ASSET_B: higher return, higher vol
    (0.07, 0.12),  # ASSET_C: lower return, lower vol (index-like)
]
daily_returns = np.column_stack([
    rng.normal(mu / TRADING_DAYS, sigma / np.sqrt(TRADING_DAYS), TRADING_DAYS)
    for mu, sigma in PARAMS
])

# Build CSV string in-memory (no file I/O needed — keeps notebook self-contained)
csv_lines = ["date," + ",".join(ASSETS)]
for i, row in enumerate(daily_returns):
    csv_lines.append(f"2025-01-{(i % 28) + 1:02d}," + ",".join(f"{v:.6f}" for v in row))
CANNED_CSV = "\n".join(csv_lines)
print(f"Canned CSV: {len(csv_lines) - 1} days × {len(ASSETS)} assets (seed=42)")
print(CANNED_CSV[:200] + "...")

## §1 — Define inputs

Load the canned CSV and compute the per-asset daily μ and σ. These become the
parameters we will propagate through the Monte Carlo loop. **State assumptions**
explicitly (reporting standard #2).

In [ ]:
# Cell 4 — Load canned CSV → per-asset (mu_daily, sigma_daily)
data = np.genfromtxt(StringIO(CANNED_CSV), delimiter=",", skip_header=1, usecols=(1, 2, 3))
mu_daily = data.mean(axis=0)            # shape (3,)
sigma_daily = data.std(axis=0, ddof=1)  # shape (3,)
print("Per-asset daily statistics (from canned data):")
for i, name in enumerate(ASSETS):
    annual_mu = mu_daily[i] * TRADING_DAYS
    annual_sigma = sigma_daily[i] * np.sqrt(TRADING_DAYS)
    print(f"  {name}: mu_daily={mu_daily[i]:.5f}  sigma_daily={sigma_daily[i]:.5f}  "
          f"| annualized mu={annual_mu:.3f}  sigma={annual_sigma:.3f}")

# Equal-weight portfolio weights (sum to 1)
WEIGHTS = np.array([1/3, 1/3, 1/3])
print(f"\nPortfolio weights (equal): {WEIGHTS}")

## §2 — Pick distribution model

We run **two** distribution models to disclose model risk (reporting standard #4):

1. **Parametric Normal** — assumes daily returns ~ N(μ, σ²). Simplest baseline; may under-estimate tails.
2. **Bootstrap (empirical)** — resample historical daily returns directly. No distributional assumption.

Comparing both shows how sensitive the risk metrics are to the distributional assumption.

In [ ]:
# Cell 6 — Vectorized MC: N paths × T timesteps, two distribution models
N_SIM = 10_000        # Iron Law: N >= 10,000 for stable tail metrics
T = TRADING_DAYS      # 1-year horizon
rng_sim = np.random.default_rng(seed=123)

def simulate_returns_parametric(mu, sigma, T, N, rng):
    """Parametric Normal: N paths × T timesteps."""
    return rng.normal(loc=mu, scale=sigma, size=(N, T, len(mu)))

def simulate_returns_bootstrap(hist, T, N, rng):
    """Bootstrap: resample historical rows with replacement."""
    idx = rng.integers(0, hist.shape[0], size=(N, T))
    return hist[idx]  # shape (N, T, n_assets)

# Parametric Normal paths
paths_param = simulate_returns_parametric(mu_daily, sigma_daily, T, N_SIM, rng_sim)
# Bootstrap paths (empirical)
paths_boot = simulate_returns_bootstrap(data, T, N_SIM, rng_sim)

# Portfolio daily return paths: weighted sum across assets
port_param = paths_param @ WEIGHTS   # shape (N, T)
port_boot  = paths_boot  @ WEIGHTS   # shape (N, T)

# Cumulative portfolio value paths (start = 1.0)
cum_param = np.cumprod(1 + port_param, axis=1)  # shape (N, T)
cum_boot  = np.cumprod(1 + port_boot,  axis=1)  # shape (N, T)

print(f"Simulated {N_SIM} paths × {T} days × 2 distribution models")
print(f"  Parametric Normal: final value mean={cum_param[:, -1].mean():.4f}  std={cum_param[:, -1].std():.4f}")
print(f"  Bootstrap (empirical): final value mean={cum_boot[:, -1].mean():.4f}  std={cum_boot[:, -1].std():.4f}")

## §3 — Compute risk metrics

For each path we compute the final return, then aggregate across the N-path
distribution into percentile bands (P5 / P50 / P95). We also compute VaR, CVaR,
and a simple Risk-Reward Ratio.

In [ ]:
# Cell 8 — Risk metrics on the distribution of N simulated outcomes
def var_cvar(pnl, alpha=0.05):
    """VaR + CVaR at confidence 1-alpha. pnl: 1D array of N P&L outcomes."""
    var = np.quantile(pnl, alpha)
    cvar = pnl[pnl <= var].mean()
    return var, cvar

def risk_reward_ratio(finals, threshold=0.0):
    """RRR = E[upside] / |E[downside]|."""
    upside = finals[finals > threshold]
    downside = finals[finals <= threshold]
    up = upside.mean() if len(upside) else 0.0
    down = abs(downside.mean()) if len(downside) else 1.0
    return up / down if down else float('inf')

def max_drawdown(cum_paths):
    """Path-dependent: worst peak-to-trough loss per path, then aggregate."""
    running_max = np.maximum.accumulate(cum_paths, axis=1)
    drawdown = (cum_paths - running_max) / running_max
    return drawdown.min(axis=1)  # most-negative per path

def summarize(finals, label):
    """Print P5/P50/P95 + risk metrics for a distribution of final returns."""
    pnl = finals - 1.0  # convert cumulative value → P&L
    p5, p50, p95 = np.percentile(finals, [5, 50, 95])
    var, cvar = var_cvar(pnl)
    rrr = risk_reward_ratio(finals)
    print(f"\n=== {label} ===")
    print(f"  Final portfolio value  P5={p5:.4f}  P50={p50:.4f}  P95={p95:.4f}")
    print(f"  VaR(5%)={var:.4f}  CVaR(5%)={cvar:.4f}  (loss thresholds on P&L)")
    print(f"  Risk-Reward Ratio={rrr:.3f}")
    return {"p5": p5, "p50": p50, "p95": p95, "var": var, "cvar": cvar, "rrr": rrr}

mdd_param = max_drawdown(cum_param)
mdd_boot  = max_drawdown(cum_boot)
summary_param = summarize(cum_param[:, -1], "Parametric Normal")
summary_boot  = summarize(cum_boot[:, -1],  "Bootstrap (empirical)")
print(f"\n  Max Drawdown  P50(param)={np.percentile(mdd_param, 50):.4f}  P50(boot)={np.percentile(mdd_boot, 50):.4f}")

## §4 — Convergence diagnostics

Check that N=10,000 is enough: the running mean of the final portfolio value
should stabilize, and doubling N should change the metric by < 1%.

In [ ]:
# Cell 10 — Running mean + doubling-N check
finals = cum_param[:, -1]
running_mean = np.cumsum(finals) / np.arange(1, len(finals) + 1)

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(running_mean, linewidth=0.8)
ax.axhline(finals.mean(), color="r", linestyle="--", label=f"final mean = {finals.mean():.4f}")
ax.set_xlabel("simulation index (sorted by run order)")
ax.set_ylabel("running mean of final value")
ax.set_title("Convergence: running mean should stabilize (Parametric Normal)")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

# Doubling-N check: compare first half vs full
half_mean = finals[: N_SIM // 2].mean()
full_mean = finals.mean()
delta_pct = abs(full_mean - half_mean) / abs(half_mean) * 100
print(f"Doubling-N check: half(N={N_SIM//2}) mean={half_mean:.5f}  full(N={N_SIM}) mean={full_mean:.5f}")
print(f"  Δ = {delta_pct:.3f}%  → {'STABLE ✓' if delta_pct < 1.0 else 'NOT STABLE — increase N ✗'}")

## §5 — Reporting standards (non-advisory)

Per skill reporting standards, we report:
1. Distribution (histogram + P5/P50/P95 band) — not a single number
2. Stated assumptions (distribution, N, horizon, data window)
3. Convergence (shown above)
4. Model-risk disclosure (Parametric vs Bootstrap side-by-side)
5. **No advice** — no buy/sell/hold/recommend language
6. **Paper-only label**

In [ ]:
# Cell 12 — Histogram + P5/P50/P95 band + model-risk comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

for ax, finals, label in [
    (axes[0], cum_param[:, -1], "Parametric Normal"),
    (axes[1], cum_boot[:, -1],  "Bootstrap (empirical)"),
]:
    ax.hist(finals, bins=80, density=True, alpha=0.7, color="steelblue", edgecolor="white")
    p5, p50, p95 = np.percentile(finals, [5, 50, 95])
    for p, c, ls in [(p5, "#d62728", "--"), (p50, "#2ca02c", "-"), (p95, "#ff7f0e", "--")]:
        ax.axvline(p, color=c, linestyle=ls, linewidth=1.6,
                   label=f"P{'5' if p==p5 else '50' if p==p50 else '95'}={p:.3f}")
    ax.set_title(f"{label}\nN={N_SIM} simulations, 1-year horizon")
    ax.set_xlabel("final portfolio value (start = 1.0)")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

axes[0].set_ylabel("density")
fig.suptitle("Monte Carlo portfolio value distribution — PAPER-ONLY, NON-ADVISORY", fontsize=11, y=1.02)
plt.tight_layout(); plt.show()

# Summary table
print("\n=== Risk-metric summary (model-risk disclosure) ===")
print(f"{'Metric':<22} {'Parametric':>12} {'Bootstrap':>12} {'Δ':>10}")
for k in ("p5", "p50", "p95", "var", "cvar"):
    a, b = summary_param[k], summary_boot[k]
    print(f"{k.upper():<22} {a:>12.4f} {b:>12.4f} {abs(a-b):>10.4f}")
print(f"{'RRR':<22} {summary_param['rrr']:>12.3f} {summary_boot['rrr']:>12.3f} {abs(summary_param['rrr']-summary_boot['rrr']):>10.3f}")
print("\n⚠️  Simulation only — not investment advice, not an order.\n"
      "    Under the stated assumptions, the simulated P5/P50/P95 of the\n"
      "    1-year portfolio value is shown above. Model risk is disclosed\n"
      "    via the Parametric-vs-Bootstrap comparison.")

## Model-risk disclosure & closing

**What this notebook shows**
- The 1-year portfolio value is a **distribution**, not a point estimate. P50 is the median outcome; P5/P95 bound the 90% uncertainty interval.
- **Model risk is real**: the Parametric Normal and Bootstrap (empirical) distributions disagree on tail shape. The Bootstrap typically shows fatter tails because it carries the empirical skew/kurtosis that the Normal assumption flattens.
- VaR/CVaR are loss thresholds on the P&L distribution; RRR compares expected upside to expected downside.

**What this notebook does NOT do**
- ❌ It does not place, modify, or cancel any real order.
- ❌ It does not read or transmit any API key, secret, or signed request.
- ❌ It does not connect to any live exchange or broker.
- ❌ It does not recommend buy / sell / hold / size — output is informational only.

**How to wire real (paper-only) data**: replace the canned CSV cell with output from the `CannedMarketDataFeed` pattern (approved 2026-06-12 amendment in `docs/protocols/bot-trading-iron-law.md`). That feed exposes only `listSymbols()` and `getKlines()` — zero write/order methods — and is the approved paper-trading data seam. The MC output above can feed a `MockBotFeed` paper-settlement workflow (simulated fills only, no real orders at any layer).

---
*Synthesized from skill `monte-carlo-quant-analysis` · sources: akashdeepo-monte-carlo-rrr, firmai-financial-machine-learning, unpingco-python-stats-ml.*